# 🚌 7790번 스마트 통학 AI - 실시간 온라인 머신러닝 학습 분석 대시보드

이 노트북은 사당역 9번 출구 및 상행 4개 정류장에서 **버스가 통과할 때마다 실시간으로 수집·학습되는 텔레메트리 데이터**와 **온라인 머신러닝 모델의 학습 과정**을 시각화합니다.

- **온라인 학습 방식**: 버스 통과 시 잔여 좌석(45/71 - 잔여석)을 즉시 역산하여 0.001초 내에 가중치 갱신
- **불금 심야 피크**: 금요일 21:30 ~ 00:40 서울 모임 귀가객 급증 패턴 반영
- **데이터 소스**: `data/sadang_telemetry.csv`, `models/sadang_online_model.json`

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime

# 한글 폰트 설정 (Windows 맑은 고딕)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

from services.online_learner import online_learner
print('✅ 시각화 및 머신러닝 환경 로드 완료!')

In [ ]:
# 1. 텔레메트리 CSV 및 학습 모델 상태 로드
SADANG_CSV = 'data/sadang_telemetry.csv'
UPSTREAM_CSV = 'data/upstream_telemetry.csv'
MODEL_JSON = 'models/sadang_online_model.json'

df_sadang = pd.read_csv(SADANG_CSV) if os.path.exists(SADANG_CSV) else pd.DataFrame()
df_upstream = pd.read_csv(UPSTREAM_CSV) if os.path.exists(UPSTREAM_CSV) else pd.DataFrame()

model_state = {}
if os.path.exists(MODEL_JSON):
    with open(MODEL_JSON, 'r', encoding='utf-8') as f:
        model_state = json.load(f)

print(f'📊 수집된 사당역 실측 차량 : {len(df_sadang)}건')
print(f'📊 수집된 상행 실측 차량   : {len(df_upstream)}건')
print(f'🤖 온라인 모델 누적 학습 대수: {model_state.get("total_learned_buses", 0)}대')
if not df_sadang.empty:
    display(df_sadang.tail(5))

### 📈 1. 사당역 시간대별 실제 탑승 인원 & 불금 심야 피크 산포도

In [ ]:
plt.figure(figsize=(12, 5))

if not df_sadang.empty:
    df_sadang['time_float'] = df_sadang['hour'] + df_sadang['minute'] / 60.0
    is_fri_night = df_sadang['is_friday_night'] == 1
    
    plt.scatter(df_sadang.loc[~is_fri_night, 'time_float'], 
                df_sadang.loc[~is_fri_night, 'actual_boarded'],
                color='#2563eb', s=90, alpha=0.75, edgecolors='black', label='평일/주말 일반 운행')
    
    plt.scatter(df_sadang.loc[is_fri_night, 'time_float'], 
                df_sadang.loc[is_fri_night, 'actual_boarded'],
                color='#e11d48', s=130, alpha=0.9, edgecolors='black', marker='^', label='🔥 불금 심야 피크 (21:30~00:40)')
else:
    plt.text(14, 25, '아직 수집된 데이터가 없습니다. 버스가 지나갈 때마다 실시간으로 점이 찍힙니다.', 
             ha='center', fontsize=12, color='gray')

plt.axhline(45, color='gray', linestyle='--', alpha=0.6, label='1층 버스 만석 기준 (45석)')
plt.axhline(71, color='purple', linestyle='--', alpha=0.6, label='2층 버스 만석 기준 (71석)')

plt.title('사당역 9번 출구 시간대별 실제 탑승 인원 분포 (GBIS 실측 텔레메트리)', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('시간대 (시)', fontsize=11)
plt.ylabel('실제 탑승 인원 (명)', fontsize=11)
plt.xlim(5.0, 25.0)
plt.ylim(0, 75)
plt.xticks(range(5, 26), [f'{h:02d}:00' if h < 24 else f'익일 {h-24:02d}:00' for h in range(5, 26)], rotation=30)
plt.legend(frameon=True, facecolor='white', framealpha=0.9)
plt.tight_layout()
plt.show()

### 🧠 2. 기존 도메인 휴리스틱(Prior) vs 실시간 온라인 ML 예측(Posterior) 곡선 비교

In [ ]:
time_slots = np.arange(5.5, 24.8, 0.25)
heuristic_curve_weekday = []
ml_curve_weekday = []
heuristic_curve_friday = []
ml_curve_friday = []

for t in time_slots:
    h = int(t)
    m = int(round((t - h) * 60))
    if m >= 60:
        h += 1
        m = 0
    
    # 일반 평일 (수요일)
    dt_wed = datetime.datetime(2026, 10, 14, h if h < 24 else 0, m)
    if 7.5 <= t < 9.0: base = 35
    elif 17.5 <= t < 19.5: base = 32
    elif 22.0 <= t < 24.5: base = 25
    else: base = 12
    heuristic_curve_weekday.append(base)
    pred_wed, _, _ = online_learner.predict_sadang_waiters(dt_wed, base, '수요일', headway_gap_min=12.0)
    ml_curve_weekday.append(pred_wed)
    
    # 금요일
    dt_fri = datetime.datetime(2026, 10, 16, h if h < 24 else 0, m)
    fri_base = int(round(base * (1.15 if t >= 21.5 else (0.70 if t < 12.0 else 1.0))))
    heuristic_curve_friday.append(fri_base)
    pred_fri, _, _ = online_learner.predict_sadang_waiters(dt_fri, fri_base, '금요일', headway_gap_min=12.0)
    ml_curve_friday.append(pred_fri)

plt.figure(figsize=(13, 5.5))
plt.plot(time_slots, heuristic_curve_weekday, 'b--', alpha=0.45, label='평일 도메인 규칙 (Prior)')
plt.plot(time_slots, ml_curve_weekday, 'b-', linewidth=2.5, label='평일 온라인 ML 학습 융합 (Posterior)')

plt.plot(time_slots, heuristic_curve_friday, 'r--', alpha=0.45, label='금요일 기본 규칙')
plt.plot(time_slots, ml_curve_friday, 'r-', linewidth=2.5, label='🔥 불금 온라인 ML 학습 융합')

plt.axvspan(7.5, 9.0, color='blue', alpha=0.08, label='등교 피크 (07:30~09:00)')
plt.axvspan(17.5, 19.5, color='green', alpha=0.08, label='퇴근 피크 (17:30~19:30)')
plt.axvspan(21.5, 24.7, color='red', alpha=0.10, label='불금 심야 피크 (21:30~00:40)')

plt.title('도메인 휴리스틱 vs 실시간 온라인 머신러닝 예측 곡선 비교', fontsize=14, fontweight='bold', pad=12)
plt.xlabel('운행 시간대', fontsize=11)
plt.ylabel('사당역 예상 대기열 (명)', fontsize=11)
plt.xlim(5.5, 24.8)
plt.ylim(0, 75)
plt.xticks(range(6, 25), [f'{h:02d}:00' for h in range(6, 25)])
plt.legend(frameon=True, facecolor='white', loc='upper left')
plt.tight_layout()
plt.show()

### 🎯 3. 누적 학습에 따른 예측 오차 (Residual) 및 MAE 수렴 추이

In [ ]:
plt.figure(figsize=(10, 4.2))

if not df_sadang.empty and 'heuristic_prior' in df_sadang.columns:
    df_sadang['residual'] = df_sadang['actual_boarded'] - df_sadang['heuristic_prior']
    df_sadang['mae_cum'] = df_sadang['residual'].abs().expanding().mean()
    
    plt.plot(range(1, len(df_sadang) + 1), df_sadang['mae_cum'], 'o-', color='#059669', linewidth=2, label='누적 평균 절대 오차 (MAE)')
    plt.axhline(4.0, color='gray', linestyle=':', label='목표 허용 오차 (±4석)')
    plt.title('온라인 학습 진행에 따른 예측 오차(MAE) 수렴 과정', fontsize=13, fontweight='bold')
    plt.xlabel('학습된 버스 대수 (대)', fontsize=11)
    plt.ylabel('평균 오차 (좌석수)', fontsize=11)
    plt.legend()
else:
    plt.text(0.5, 0.5, '아직 누적된 학습 데이터가 1건 이하입니다.\n버스가 운행되면서 오차가 수렴하는 곡선이 시각화됩니다.', 
             ha='center', va='center', fontsize=12, color='gray')
    plt.title('온라인 학습 오차 수렴 추이', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 🎛️ 4. 실시간 온라인 예측 인터랙티브 시뮬레이터

In [ ]:
def test_predict(weekday_name='금요일', hour=22, minute=30, headway_min=12.0):
    weekday_map = {'월요일': 0, '화요일': 1, '수요일': 2, '목요일': 3, '금요일': 4, '토요일': 5, '일요일': 6}
    w_idx = weekday_map.get(weekday_name, 4)
    dt = datetime.datetime(2026, 10, 12 + w_idx, hour, minute)
    
    t_float = hour + minute / 60.0
    if w_idx == 4 and t_float >= 21.5:
        base = 48
    elif 7.5 <= t_float <= 9.0:
        base = 35
    elif 17.5 <= t_float <= 19.5:
        base = 32
    else:
        base = 15
        
    est, desc, meta = online_learner.predict_sadang_waiters(
        now_dt=dt,
        heuristic_base=base,
        heuristic_desc=f'{weekday_name} {hour:02d}:{minute:02d}',
        headway_gap_min=headway_min
    )
    
    print(f'[{weekday_name} {hour:02d}:{minute:02d} (배차간격 {headway_min}분) 시뮬레이션]')
    print(f'  • 도메인 기본값 : {base}명')
    print(f'  • 최종 예측 인원: {est}명 (차이: {est - base:+d}명)')
    print(f'  • ML 반영 가중치: {meta.get("mlBlendWeight", 0)}%')
    print(f'  • 모델 상세 설명: {desc}')

# 시뮬레이션 테스트: 불금 밤 22:45
test_predict('금요일', 22, 45, headway_min=14.0)